In [ ]:
# ============================================================
# BOOTSTRAP CELL -- run this first.
# Installs anything missing, detects CUDA / MPS / CPU, wires the
# local sam2/ and build.py into sys.path. Works on Windows,
# Linux, macOS, and Google Colab without any extra step.
# ============================================================
import importlib, shutil, subprocess, sys, os, platform
from pathlib import Path

# 1) locate the bundle root (walk up until we find build.py)
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'build.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'build.py').exists():
    raise RuntimeError('Could not find build.py by walking up from the notebook folder. '
                       'Open the notebook from inside the bundle folder.')
os.environ['SAM2_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)

# 2) helper -- tries system pip first, falls back to --user
def _pip(pkgs, extra=()):
    cmd = [sys.executable, '-m', 'pip', 'install', *pkgs, *extra]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', *pkgs, *extra])

# 3) make sure torch + torchvision are there with the right wheel
def _detect_cuda():
    exe = shutil.which('nvidia-smi')
    if not exe: return None
    try:
        out = subprocess.check_output([exe], stderr=subprocess.STDOUT, timeout=8).decode(errors='ignore')
        for line in out.splitlines():
            if 'CUDA Version' in line:
                return line.split('CUDA Version:')[1].strip().split()[0]
    except Exception:
        pass
    return None

try:
    import torch  # noqa: F401
except ImportError:
    sys_name = platform.system()
    in_colab = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
    if in_colab:
        pass  # Colab already ships torch with CUDA
    elif _detect_cuda() and sys_name in ('Windows', 'Linux'):
        print('NVIDIA GPU detected -> installing CUDA torch wheel ...')
        _pip(['torch', 'torchvision'], extra=['--index-url', 'https://download.pytorch.org/whl/cu121'])
    elif sys_name == 'Darwin':
        print('macOS detected -> installing default torch (MPS/CPU) ...')
        _pip(['torch', 'torchvision'])
    else:
        print('No GPU -> installing CPU-only torch ...')
        _pip(['torch', 'torchvision'], extra=['--index-url', 'https://download.pytorch.org/whl/cpu'])

# 4) install everything else that is missing
_needed = {
    'hydra':            'hydra-core',
    'omegaconf':        'omegaconf',
    'iopath':           'iopath',
    'huggingface_hub':  'huggingface_hub',
    'cv2':              'opencv-python',
    'matplotlib':       'matplotlib',
    'ipympl':           'ipympl',
    'PIL':              'pillow',
    'tqdm':             'tqdm',
}
_missing = []
for mod, pkg in _needed.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        _missing.append(pkg)
if _missing:
    print('Installing missing packages:', _missing)
    _pip(_missing)
    print('If any import still fails below, restart the kernel and rerun this cell.')

# 5) report the torch backend
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
    os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
else:
    device = torch.device('cpu')
print('torch', torch.__version__, '| device:', device)


In [ ]:
%matplotlib widget


In [ ]:
import os
from pathlib import Path
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import pickle
from matplotlib.widgets import Button, RadioButtons
import cv2
import sys
import copy

import importlib
import build
importlib.reload(build)
from functions.automatic_mask_generator import SAM2AutomaticMaskGenerator
from functions.sam2_image_predictor import SAM2ImagePredictor

PROJECT_ROOT = Path(os.environ.get("SAM2_PROJECT_ROOT", Path.cwd()))
CONFIG_DIR = PROJECT_ROOT / "configs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"



In [ ]:

def show_anns(anns, ax, colors, borders=True):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:, :, 3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_id = ann['id'] % len(colors)
        color_mask = np.concatenate([colors[color_id], [0.5]])
        img[m] = color_mask
        if borders:
            contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0, 0, 1, 0.4), thickness=1)
    ax.imshow(img)

def generate_overlaid_image(image, anns, colors):
    image_uint8 = image if image.dtype == np.uint8 else (image * 255).astype(np.uint8)
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    overlay = np.zeros((image.shape[0], image.shape[1], 4), dtype=np.float32)
    overlay[:, :, 3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_id = ann['id'] % len(colors)
        color_mask = np.concatenate([colors[color_id], [0.5]])
        overlay[m] = color_mask
    alpha = overlay[:, :, 3:4]
    blended = (image_uint8.astype(np.float32) / 255.0) * (1 - alpha) + overlay[:, :, :3] * alpha
    blended_uint8 = (blended * 255).astype(np.uint8)
    # Draw borders
    for ann in sorted_anns:
        m = ann['segmentation']
        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        cv2.drawContours(blended_uint8, contours, -1, (0, 0, 255), thickness=1)  # Blue in RGB
    return blended_uint8

MODEL_FILES = {
    'tiny': ("sam2.1_hiera_t.yaml", "sam2.1_hiera_tiny.pt"),
    'small': ("sam2.1_hiera_s.yaml", "sam2.1_hiera_small.pt"),
    'base_plus': ("sam2.1_hiera_b+.yaml", "sam2.1_hiera_base_plus.pt"),
    'large': ("sam2.1_hiera_l.yaml", "sam2.1_hiera_large.pt"),
}

class InteractiveMaskPanel:
    def __init__(self, directory, index):
        self.directory = directory
        self.index = index
        self.load_current()
        self.model_name = 'large'  # Default model
        self.load_model(self.model_name)
        self.predictor.set_image(self.image)  # Set image for predictor
        self.mode = 'remove'  # Default mode: 'remove' or 'add'
        self.points = []  # For add mode: list of (x, y, label) where label=1 positive, 0 negative
        self.box = None  # For add mode: [x0, y0, x1, y1]
        self.box_start = None
        self.drawing_box = False
        self.bg_mode = 0  # 0: raw + colored masks, 1: black + colored masks, 2: raw + blue union overlay
        self.view_raw = False  # New toggle for full raw view
        self.assign_ids()  # Assign IDs if not present
        self.next_id = max([mask['id'] for mask in self.masks]) + 1 if self.masks else 0
        self.colors = [np.random.random(3) for _ in range(10000)]  # Fixed random colors for IDs 0 to 9999
        self.undo_stack = []  # Stack for undo actions
        self.fig = plt.figure(figsize=(12, 12))
        self.ax = self.fig.add_subplot(1, 1, 1)
        plt.subplots_adjust(bottom=0.3, left=0.1, right=0.9, top=0.85)  # Make space for buttons and radio

        # Buttons
        self.ax_add = plt.axes([0.05, 0.2, 0.08, 0.075])
        self.btn_add = Button(self.ax_add, 'Add')
        self.btn_add.label.set_fontsize(8)
        self.btn_add.on_clicked(self.on_add_mode)

        self.ax_remove = plt.axes([0.15, 0.2, 0.08, 0.075])
        self.btn_remove = Button(self.ax_remove, 'Remove')
        self.btn_remove.label.set_fontsize(8)
        self.btn_remove.on_clicked(self.on_remove_mode)

        self.ax_draw_box = plt.axes([0.25, 0.2, 0.08, 0.075])
        self.btn_draw_box = Button(self.ax_draw_box, 'Box')
        self.btn_draw_box.label.set_fontsize(8)
        self.btn_draw_box.on_clicked(self.on_draw_box)

        self.ax_generate = plt.axes([0.35, 0.2, 0.08, 0.075])
        self.btn_generate = Button(self.ax_generate, 'Mask')
        self.btn_generate.label.set_fontsize(8)
        self.btn_generate.on_clicked(self.on_generate)

        self.ax_clear = plt.axes([0.45, 0.2, 0.08, 0.075])
        self.btn_clear = Button(self.ax_clear, 'Clear')
        self.btn_clear.label.set_fontsize(8)
        self.btn_clear.on_clicked(self.on_clear)

        self.ax_toggle_bg = plt.axes([0.55, 0.2, 0.08, 0.075])
        self.btn_toggle_bg = Button(self.ax_toggle_bg, 'Bkg')
        self.btn_toggle_bg.label.set_fontsize(8)
        self.btn_toggle_bg.on_clicked(self.on_toggle_bg)

        self.ax_raw_toggle = plt.axes([0.65, 0.2, 0.08, 0.075])
        self.btn_raw_toggle = Button(self.ax_raw_toggle, 'Raw')
        self.btn_raw_toggle.label.set_fontsize(8)
        self.btn_raw_toggle.on_clicked(self.on_raw_toggle)

        self.ax_erase_all = plt.axes([0.75, 0.2, 0.08, 0.075])
        self.btn_erase_all = Button(self.ax_erase_all, 'Erase All')
        self.btn_erase_all.label.set_fontsize(8)
        self.btn_erase_all.on_clicked(self.on_erase_all)

        self.ax_back = plt.axes([0.05, 0.05, 0.08, 0.075])
        self.btn_back = Button(self.ax_back, 'Back')
        self.btn_back.label.set_fontsize(8)
        self.btn_back.on_clicked(self.on_back)

        self.ax_next = plt.axes([0.15, 0.05, 0.08, 0.075])
        self.btn_next = Button(self.ax_next, 'Next')
        self.btn_next.label.set_fontsize(8)
        self.btn_next.on_clicked(self.on_next)

        self.ax_save = plt.axes([0.25, 0.05, 0.08, 0.075])
        self.btn_save = Button(self.ax_save, 'Save')
        self.btn_save.label.set_fontsize(8)
        self.btn_save.on_clicked(self.on_save)

        self.ax_quit = plt.axes([0.35, 0.05, 0.08, 0.075])
        self.btn_quit = Button(self.ax_quit, 'Quit')
        self.btn_quit.label.set_fontsize(8)
        self.btn_quit.on_clicked(self.on_quit)

        self.ax_undo = plt.axes([0.45, 0.05, 0.08, 0.075])
        self.btn_undo = Button(self.ax_undo, 'Undo')
        self.btn_undo.label.set_fontsize(8)
        self.btn_undo.on_clicked(self.on_undo)

        # Model selection radio buttons - moved to top
        self.rax = plt.axes([0.4, 0.9, 0.2, 0.1])
        self.radio = RadioButtons(self.rax, ('tiny', 'small', 'base_plus', 'large'))
        self.radio.set_active(3)  # Set active before attaching callback
        self.radio.on_clicked(self.on_model_change)

        self.fig.canvas.mpl_connect('button_press_event', self.on_click)
        # Optionally keep key press if it works in some setups
        self.fig.canvas.mpl_connect('key_press_event', self.on_key)

        self.redraw()
        print("Instructions:\n"
              "- Use radio buttons on the top to select model (tiny, small, base_plus, large); switching updates the predictor without regenerating masks.\n"
              "- Use buttons at the bottom to switch modes, draw box, generate mask, clear, toggle background (raw + colored masks, masks only on black, raw + blue union overlay), or quit.\n"
              "- In remove mode: Click on a mask to remove it (all overlapping at the point). You can click multiple times for removals.\n"
              "- In add mode: Left-click for positive point (green), right-click for negative point (red). You can generate multiple masks.\n"
              "- For box: Click 'Draw Box' button, then left-click start and end points on image (click 'Draw Box' again to cancel).\n"
              "- After setting points and/or box, click 'Generate Mask' to add new mask (requires at least one positive point or box).\n"
              "- Changes are accepted automatically. Use 'Undo' to revert the last action (add or remove).\n"
              "- Click 'Erase All' to remove all masks (can be undone).\n"
              "- Click 'Save' to save current to pickle and PNG without loading next.\n"
              "- Click 'Next' to save and load next.\n"
              "- Click 'Back' to load previous image.\n"
              "- Key presses are also supported as fallback if they work in your environment: a: add mode, m: remove mode, d: draw box, enter: generate, c: clear, t: toggle bg, r: raw, n: next, b: back, s: save, e: erase all, q: quit, u: undo.")
    def assign_ids(self):
        current_id = 0
        for mask in self.masks:
            if 'id' not in mask:
                mask['id'] = current_id
                current_id += 1
    def load_current(self):
        filename = f'B{self.index:04d}.png'
        pickle_filename = f'B{self.index:04d}_masks.pickle'
        image_path = os.path.join(self.directory, filename)
        pickle_path = os.path.join(self.directory, pickle_filename)
        if os.path.exists(image_path):
            self.image = np.array(Image.open(image_path).convert("RGB"))
        else:
            raise FileNotFoundError(f"Image not found: {image_path}")
        if os.path.exists(pickle_path):
            with open(pickle_path, 'rb') as f:
                self.masks = pickle.load(f)
        else:
            self.masks = []
    def save_current(self, save_png=False):
        pickle_filename = f'B{self.index:04d}_masks.pickle'
        pickle_path = os.path.join(self.directory, pickle_filename)
        with open(pickle_path, 'wb') as f:
            pickle.dump(self.masks, f)
        print(f'Saved updated masks to {pickle_path}')
        if save_png:
            overlaid_image = generate_overlaid_image(self.image, self.masks, self.colors)
            png_filename = f'B{self.index:04d}_segmented.png'
            png_path = os.path.join(self.directory, png_filename)
            Image.fromarray(overlaid_image).save(png_path)
            print(f'Saved segmented PNG to {png_path}')
    def load_model(self, model_name):
        if model_name not in MODEL_FILES:
            raise ValueError(f"Invalid model name: {model_name}")
        config_name, checkpoint_name = MODEL_FILES[model_name]
        model_cfg = CONFIG_DIR / config_name
        sam2_checkpoint = CHECKPOINT_DIR / checkpoint_name
        if not model_cfg.exists():
            raise FileNotFoundError(f"Config file not found: {model_cfg}")
        if not sam2_checkpoint.exists():
            raise FileNotFoundError(f"Checkpoint file not found: {sam2_checkpoint}")

        self.sam2_model = build.build_segment(
            config_file=str(model_cfg),
            ckpt_path=str(sam2_checkpoint),
            device=device,
        )
        self.mask_generator = SAM2AutomaticMaskGenerator(
            model=self.sam2_model,
            points_per_side=128,
            points_per_batch=128,
            pred_iou_thresh=0.7,
            stability_score_thresh=0.92,
            stability_score_offset=0.7,
            crop_n_layers=1,
            box_nms_thresh=0.7,
            crop_n_points_downscale_factor=2,
            min_mask_region_area=25.0,
            use_m2m=True,
        )
        self.predictor = SAM2ImagePredictor(self.sam2_model)
    def on_model_change(self, label):
        self.model_name = label
        self.load_model(self.model_name)
        self.predictor.set_image(self.image)
        self.next_id = max([mask['id'] for mask in self.masks]) + 1 if self.masks else 0  # Reset next_id based on current masks
        self.redraw()
    def redraw(self):
        self.ax.clear()
        if self.view_raw:
            self.ax.imshow(self.image)
            self.ax.set_title("Raw Image")
        else:
            if self.bg_mode == 2:
                # Compute blue union overlay
                binary_mask = np.zeros(self.image.shape[:2], dtype=bool)
                if self.masks:
                    for ann in self.masks:
                        binary_mask |= ann['segmentation']
                overlay = np.zeros((self.image.shape[0], self.image.shape[1], 4), dtype=np.float32)
                overlay[:, :, 3] = 0
                if np.any(binary_mask):
                    color_mask = np.array([0, 0, 1, 0.3])  # Blue with alpha 0.3
                    overlay[binary_mask] = color_mask
                alpha = overlay[:, :, 3:]
                blended = (self.image.astype(np.float32) / 255.0) * (1 - alpha) + overlay[:, :, :3] * alpha
                blended_uint8 = (blended * 255).astype(np.uint8)
                self.ax.imshow(blended_uint8)
                # Add points and box if any
                if self.points:
                    pos_points = np.array([p[:2] for p in self.points if p[2] == 1])
                    neg_points = np.array([p[:2] for p in self.points if p[2] == 0])
                    if len(pos_points) > 0:
                        self.ax.scatter(pos_points[:, 0], pos_points[:, 1], c='g', marker='*', s=100)
                    if len(neg_points) > 0:
                        self.ax.scatter(neg_points[:, 0], neg_points[:, 1], c='r', marker='*', s=100)
                if self.box is not None:
                    x0, y0, x1, y1 = self.box
                    self.ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor='green', facecolor='none', lw=2))
                title = f"Mode: {self.mode} | Model: {self.model_name} | Masks: {len(self.masks)} | File: B{self.index:04d}.png | View: Blue Union Overlay"
            else:
                if self.bg_mode == 0:
                    self.ax.imshow(self.image)
                else:
                    self.ax.imshow(np.zeros_like(self.image))
                show_anns(self.masks, self.ax, self.colors)
                if self.points:
                    pos_points = np.array([p[:2] for p in self.points if p[2] == 1])
                    neg_points = np.array([p[:2] for p in self.points if p[2] == 0])
                    if len(pos_points) > 0:
                        self.ax.scatter(pos_points[:, 0], pos_points[:, 1], c='g', marker='*', s=100)
                    if len(neg_points) > 0:
                        self.ax.scatter(neg_points[:, 0], neg_points[:, 1], c='r', marker='*', s=100)
                if self.box is not None:
                    x0, y0, x1, y1 = self.box
                    self.ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor='green', facecolor='none', lw=2))
                title = f"Mode: {self.mode} | Model: {self.model_name} | Masks: {len(self.masks)} | File: B{self.index:04d}.png"
                if self.bg_mode == 1:
                    title += " | View: Masks Only"
            self.ax.set_title(title)
        self.fig.canvas.draw()
    def on_raw_toggle(self, event):
        self.view_raw = not self.view_raw
        self.redraw()
    def on_click(self, event):
        if event.inaxes != self.ax:  # Ignore clicks on buttons and raw axes
            return
        x, y = int(event.xdata), int(event.ydata)
        if self.mode == 'remove':
            to_remove = []
            for i, mask in enumerate(self.masks):
                if mask['segmentation'][y, x]:
                    to_remove.append(i)
            deleted = []
            for i in sorted(to_remove, reverse=True):
                deleted.append(self.masks.pop(i))
            if deleted:
                self.undo_stack.append(('remove', copy.deepcopy(deleted)))
            self.redraw()
        elif self.mode == 'add':
            if self.drawing_box:
                if self.box_start is None:
                    self.box_start = (x, y)
                    print("Start point set. Click for end point or click 'Draw Box' to cancel.")
                else:
                    x_start, y_start = self.box_start
                    self.box = [min(x_start, x), min(y_start, y), max(x_start, x), max(y_start, y)]
                    self.box_start = None
                    self.drawing_box = False
                    print("Box set.")
                    self.redraw()
            else:
                if event.button == 1:  # Left click: positive point
                    self.points.append((x, y, 1))
                elif event.button == 3:  # Right click: negative point
                    self.points.append((x, y, 0))
                self.redraw()
    def on_key(self, event):  # Kept as fallback
        if event.key == 'a':
            self.on_add_mode(None)
        elif event.key == 'm':
            self.on_remove_mode(None)
        elif event.key == 'd':
            self.on_draw_box(None)
        elif event.key == 'enter':
            self.on_generate(None)
        elif event.key == 'c':
            self.on_clear(None)
        elif event.key == 't':
            self.on_toggle_bg(None)
        elif event.key == 'e':
            self.on_erase_all(None)
        elif event.key == 'q':
            self.on_quit(None)
        elif event.key == 'u':
            self.on_undo(None)
        elif event.key == 'r':
            self.on_raw_toggle(None)
        elif event.key == 'n':
            self.on_next(None)
        elif event.key == 'b':
            self.on_back(None)
        elif event.key == 's':
            self.on_save(None)
    def on_add_mode(self, event):
        self.mode = 'add'
        self.points = []
        self.box = None
        self.box_start = None
        self.drawing_box = False
        self.redraw()
    def on_remove_mode(self, event):
        self.mode = 'remove'
        self.points = []
        self.box = None
        self.box_start = None
        self.drawing_box = False
        self.redraw()
    def on_draw_box(self, event):
        if self.mode == 'add':
            if self.drawing_box:
                self.drawing_box = False
                self.box_start = None
                print("Box drawing cancelled.")
            else:
                self.drawing_box = True
                self.box_start = None
                print("Box drawing started. Click for start point.")
            self.redraw()
    def on_generate(self, event):
        if self.mode == 'add' and (self.points or self.box is not None):
            point_coords = np.array([p[:2] for p in self.points]) if self.points else None
            point_labels = np.array([p[2] for p in self.points]) if self.points else None
            box = np.array(self.box) if self.box else None
            # Debug print
            print(f"Generating mask with points: {len(self.points) if self.points else 0}, box: {box}")
            masks, scores, _ = self.predictor.predict(
                point_coords=point_coords if point_coords is not None and len(point_coords) > 0 else None,
                point_labels=point_labels if point_labels is not None and len(point_labels) > 0 else None,
                box=box,
                multimask_output=False  # Single mask output
            )
            print(f"Predicted {masks.shape[0]} masks")
            num_added = 0
            if masks.shape[0] > 0:
                if torch.is_tensor(masks):
                    masks = (masks > 0.0).cpu().numpy()
                else:
                    masks = (masks > 0.0).astype(np.bool_)
                new_mask = {
                    'segmentation': masks[0],
                    'area': np.sum(masks[0]),
                    'bbox': self.box if self.box else [0, 0, 0, 0],  # Placeholder
                    'predicted_iou': scores[0].item() if scores.size > 0 else 0.0,
                    'point_coords': [point_coords.tolist()] if point_coords is not None else [[0, 0]],
                    'stability_score': 0.95,  # Placeholder
                    'crop_box': [0, 0, self.image.shape[1], self.image.shape[0]],
                    'id': self.next_id
                }
                self.next_id += 1
                self.masks.append(new_mask)
                print("New mask added.")
                num_added = 1
            else:
                print("No mask generated. Ensure at least one positive point or a box is provided.")
            if num_added > 0:
                self.undo_stack.append(('add', num_added))
            self.points = []
            self.box = None
            self.box_start = None
            self.drawing_box = False
            self.redraw()
    def on_clear(self, event):
        if self.mode == 'add':
            self.points = []
            self.box = None
            self.box_start = None
            self.drawing_box = False
            self.redraw()
    def on_erase_all(self, event):
        if len(self.masks) > 0:
            self.undo_stack.append(('remove', copy.deepcopy(self.masks)))
            self.masks = []
            self.redraw()
    def on_undo(self, event):
        if self.undo_stack:
            action, data = self.undo_stack.pop()
            if action == 'add':
                for _ in range(data):
                    if self.masks:
                        self.masks.pop()
                        self.next_id -= 1
            elif action == 'remove':
                self.masks.extend(copy.deepcopy(data))
            print("Undid last action.")
            self.redraw()
        else:
            print("Nothing to undo.")
    def on_toggle_bg(self, event):
        self.bg_mode = (self.bg_mode + 1) % 3
        self.redraw()
    def load_next(self, save_current_first=False):
        if save_current_first:
            self.save_current(save_png=True)
        self.index += 1
        try:
            self.load_current()
            self.predictor.set_image(self.image)
            self.assign_ids()
            self.next_id = max([mask['id'] for mask in self.masks]) + 1 if self.masks else 0
            self.points = []
            self.box = None
            self.box_start = None
            self.drawing_box = False
            self.mode = 'remove'  # Reset to default mode
            self.undo_stack = []
            self.redraw()
        except FileNotFoundError:
            print(f"No more images found after index {self.index - 1}. Quitting.")
            self.on_quit(None)
    def on_save(self, event):
        self.save_current(save_png=True)
    def on_next(self, event):
        self.load_next(save_current_first=True)
    def load_back(self):
        if self.index > 1:
            self.index -= 1
            self.load_current()
            self.predictor.set_image(self.image)
            self.assign_ids()
            self.next_id = max([mask['id'] for mask in self.masks]) + 1 if self.masks else 0
            self.points = []
            self.box = None
            self.box_start = None
            self.drawing_box = False
            self.mode = 'remove'  # Reset to default mode
            self.undo_stack = []
            self.redraw()
    def on_back(self, event):
        self.load_back()
    def on_quit(self, event):
        self.save_current(save_png=True)
        plt.close(self.fig)
        print("Final masks saved. Access via panel.masks if needed.")


In [ ]:
# IMPORTANT: Run in a Jupyter cell: %matplotlib widget
from pathlib import Path
import os
PROJECT_ROOT = Path(os.environ.get('SAM2_PROJECT_ROOT', Path.cwd())).resolve()
directory = str(PROJECT_ROOT / 'pkl')
start_index = 1  # Start from B0001
panel = InteractiveMaskPanel(directory, start_index)
